# ETH/USD September Analysis

This notebook fetches ETH/USD price data for September 2024, performs exploratory data analysis, and generates a 14-day ARIMA forecast.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from statsmodels.tsa.arima.model import ARIMA


In [ ]:
# Fetch ETH/USD data for September 2024
eth = yf.download('ETH-USD', start='2024-09-01', end='2024-10-01')
os.makedirs('data', exist_ok=True)
eth.to_csv('data/eth_sep.csv')
eth.head()


In [ ]:
# Plot closing price with moving averages
eth['MA7'] = eth['Close'].rolling(window=7).mean()
eth['MA30'] = eth['Close'].rolling(window=30).mean()
os.makedirs('outputs', exist_ok=True)
plt.figure(figsize=(10,6))
plt.plot(eth.index, eth['Close'], label='Close')
plt.plot(eth.index, eth['MA7'], label='7-day MA')
plt.plot(eth.index, eth['MA30'], label='30-day MA')
plt.title('ETH/USD Closing Price with Moving Averages')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.tight_layout()
plt.savefig('outputs/eth_price.png')
plt.show()


In [ ]:
# Plot daily volume
plt.figure(figsize=(10,4))
plt.bar(eth.index, eth['Volume'])
plt.title('ETH/USD Daily Volume')
plt.xlabel('Date')
plt.ylabel('Volume')
plt.tight_layout()
plt.savefig('outputs/eth_volume.png')
plt.show()


In [ ]:
# Compute daily returns and stats
eth['Return'] = eth['Close'].pct_change()
mean_ret = eth['Return'].mean()
std_ret = eth['Return'].std()
# Max drawdown
cum_return = (1 + eth['Return']).cumprod()
roll_max = cum_return.cummax()
drawdown = (cum_return - roll_max) / roll_max
max_drawdown = drawdown.min()
print(f"Mean: {mean_ret:.4f}, Std: {std_ret:.4f}, Max Drawdown: {max_drawdown:.4f}")
plt.figure(figsize=(8,5))
plt.hist(eth['Return'].dropna(), bins=20, edgecolor='k')
plt.title('ETH/USD Daily Returns')
plt.xlabel('Return')
plt.ylabel('Frequency')
plt.tight_layout()
plt.savefig('outputs/eth_returns_hist.png')
plt.show()


In [ ]:
# ARIMA forecasting for next 14 days
model = ARIMA(eth['Close'], order=(1,1,1))
model_fit = model.fit()
forecast = model_fit.get_forecast(steps=14)
forecast_df = forecast.summary_frame()
plt.figure(figsize=(10,6))
plt.plot(eth.index, eth['Close'], label='Observed')
forecast_index = forecast_df.index
plt.plot(forecast_index, forecast_df['mean'], label='Forecast')
plt.fill_between(forecast_index, forecast_df['mean_ci_lower'], forecast_df['mean_ci_upper'], color='pink', alpha=0.3)
plt.title('ETH/USD 14-day Forecast')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.tight_layout()
plt.savefig('outputs/eth_forecast.png')
plt.show()


## Summary

- `data/eth_sep.csv` contains the raw September data.
- Plots are saved in the `outputs/` directory.
- The ARIMA model provides a simple 14-day forecast with confidence intervals.
